# 001c — Extract HeAR Embeddings + Clinical Features (v3)

特徵向量（520 維）：
- `emb_0` ~ `emb_511`：HeAR embedding（512維）
- `age`：年齡（數值）
- `gender_encoded`：性別（male=0, female=1, other=2, unknown=-1）
- `respiratory_condition`：慢性呼吸道疾病（0/1）
- `fever_muscle_pain`：發燒或肌肉痠痛（0/1）
- `cough_type_encoded`：咳嗽類型，wet=1 / dry=0 / unknown=-1（新增）
- `dyspnea`：呼吸困難，1/0/-1（新增）
- `wheezing`：喘鳴聲，1/0/-1（新增）
- `congestion`：鼻塞，1/0/-1（新增）

## Imports

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import librosa
import tensorflow as tf
from huggingface_hub import from_pretrained_keras, notebook_login
from huggingface_hub.utils import HfFolder

OUTPUT_DIR   = 'data'
SAMPLE_RATE  = 16000
CLIP_SAMPLES = SAMPLE_RATE * 2

c:\Users\aint\anaconda3\envs\base1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 登入 Hugging Face

In [2]:
if HfFolder.get_token() is None:
    notebook_login()
else:
    print('已登入 Hugging Face')

已登入 Hugging Face


## 載入 HeAR

In [3]:
print('載入 HeAR 模型中...')
hear_model = from_pretrained_keras('google/hear')
serving_fn  = hear_model.signatures['serving_default']
print('HeAR 載入完成！')

載入 HeAR 模型中...


Fetching 24 files: 100%|██████████| 24/24 [00:00<?, ?it/s]

HeAR 載入完成！


## 音訊切片函式

In [4]:
def load_and_slice(file_path, sr=SAMPLE_RATE, clip_samples=CLIP_SAMPLES):
    audio, _ = librosa.load(file_path, sr=sr, mono=True)
    if len(audio) < clip_samples:
        audio = np.pad(audio, (0, clip_samples - len(audio)))
    n_clips = len(audio) // clip_samples
    return np.array([audio[i*clip_samples:(i+1)*clip_samples]
                     for i in range(n_clips)], dtype=np.float32)

def get_embedding(file_path):
    clips = load_and_slice(file_path)
    if len(clips) == 0:
        return None
    emb = serving_fn(x=tf.constant(clips))['output_0'].numpy()
    return emb.mean(axis=0)

## 提取 Embedding & 合併臨床特徵

In [5]:
df_list = pd.read_csv('data/prepared_train_coughvid_list.csv')
print(f'總樣本數: {len(df_list)}')
print(df_list['label'].value_counts())

embedding_rows = []
errors = []

for _, row in tqdm(df_list.iterrows(), total=len(df_list)):
    try:
        emb = get_embedding(row['file_path'])
        if emb is None:
            errors.append({'file': row['file_path'], 'error': 'audio too short'})
            continue

        entry = {f'emb_{i}': emb[i] for i in range(512)}

        # 四項原有臨床特徵
        entry['age']                   = float(row['age'])
        entry['gender_encoded']        = int(row['gender_encoded'])
        entry['respiratory_condition'] = int(row['respiratory_condition'])
        entry['fever_muscle_pain']     = int(row['fever_muscle_pain'])

        # 四項新增 expert 臨床特徵（-1 = unknown/未填）
        entry['cough_type_encoded'] = int(row.get('cough_type_encoded', -1))
        entry['dyspnea']            = int(row.get('dyspnea', -1))
        entry['wheezing']           = int(row.get('wheezing', -1))
        entry['congestion']         = int(row.get('congestion', -1))

        entry['filename'] = row['filename']
        entry['label']    = row['label']
        embedding_rows.append(entry)

    except Exception as e:
        errors.append({'file': row['file_path'], 'error': str(e)})

print(f'\n成功: {len(embedding_rows)} 筆，失敗: {len(errors)} 筆')

總樣本數: 2100
label
healthy        700
covid          700
symptomatic    700
Name: count, dtype: int64


100%|██████████| 2100/2100 [09:29<00:00,  3.69it/s]


成功: 2100 筆，失敗: 0 筆


## 輸出 CSV

In [6]:
EMB_COLS  = [f'emb_{i}' for i in range(512)]
CLIN_COLS = ['age', 'gender_encoded', 'respiratory_condition', 'fever_muscle_pain',
             'cough_type_encoded', 'dyspnea', 'wheezing', 'congestion']  # 新增四欄
ALL_COLS  = EMB_COLS + CLIN_COLS + ['filename', 'label']

df_embeddings = pd.DataFrame(embedding_rows).reindex(columns=ALL_COLS)

out_path = os.path.join(OUTPUT_DIR, 'prepared_train_hear.csv')
df_embeddings.to_csv(out_path, index=False)

print(f'已儲存至 {out_path}')
print(f'特徵維度：512 (HeAR) + 8 (clinical) = 520 維')
print(df_embeddings['label'].value_counts())
df_embeddings[['filename','label'] + CLIN_COLS].head(10)

已儲存至 data\prepared_train_hear.csv
特徵維度：512 (HeAR) + 8 (clinical) = 520 維
label
healthy        700
covid          700
symptomatic    700
Name: count, dtype: int64


,filename,label,age,gender_encoded,respiratory_condition,fever_muscle_pain,cough_type_encoded,dyspnea,wheezing,congestion
0,e27b104a-e2fd-403e-a02f-ffd0560cd52a.wav,healthy,19.0,0,0,0,0,0,0,0
1,7fe5ab80-77f1-4777-b5b2-7c239608f6b5.wav,healthy,42.0,1,1,0,-1,0,0,0
2,25d5e8bc-0203-4909-aebb-00df700417b9.wav,covid,31.0,1,0,1,0,0,0,0
3,bbe5b8d6-21cf-4718-b263-0fb779d13784.wav,healthy,35.0,1,1,1,-1,0,0,0
4,0b618a49-9562-4fa5-afb7-fce583b5a444.wav,covid,22.0,1,1,0,0,0,0,0
5,21601275-9606-4574-869d-1cec6120d3e1.wav,symptomatic,46.0,0,1,1,0,0,0,0
6,e6b481ff-0920-4d48-bb9d-8b5349c483b7.wav,healthy,35.0,1,0,0,-1,0,0,0
7,aug_5aa9d0aa-28d8-477c-8fd1-e59beb9823f0_noise...,covid,24.0,0,0,0,-1,0,0,0
8,aug_c9c4445b-fbd3-4f4d-9ca2-282a669d79c4_pitch...,symptomatic,38.0,0,1,0,0,0,0,0
9,d1c5cabc-edf9-4244-9ba4-a379bf154e29.wav,symptomatic,41.0,1,0,0,0,0,0,0


## 失敗檔案檢查（選用）

In [7]:
if errors:
    print(pd.DataFrame(errors).to_string())
else:
    print('沒有失敗的檔案！')

沒有失敗的檔案！
